<a href="https://colab.research.google.com/github/Aaditya-22/pyt-classifier/blob/dog-cat-classifier/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                         import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import os
from PIL import Image

In [ ]:
#cleaning data

path= 'PetImages'

for folder in os.listdir(path):
  for img_file in os.listdir(os.path.join(path,folder)):
    img_file= os.path.join(path,folder,img_file)

    try:
      img=Image.open(img_file)
      if img.mode!='RGB':
        os.remove(img_file)
    except:
      os.remove(img_file)

FileNotFoundError: [Errno 2] No such file or directory: 'PetImages'

In [ ]:
# pre-process data

transform= transforms.Compose([
                              transforms.Resize(255),
                              transforms.CenterCrop(224),
                              transforms.ToTensor(),
                              transforms.Normalize([0.5], [0.5])
                              ])
dataset= datasets.ImageFolder('PetImages', transform=transform)

dataset_len=len(dataset)

train_len, test_len= dataset_len-6000, 6000
train_set,test_set=torch.utils.data.random_split(dataset,[train_len,test_len])
batch_size=200

train_set= DataLoader(dataset=train_set, shuffle=True, batch_size=batch_size)
test_set= DataLoader(dataset=test_set, shuffle=True, batch_size=batch_size)


device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Using device:',device)

Using device: cuda


In [ ]:
#cnn model

class Model(torch.nn.Module):
  def __init__(self):
    super(Model, self).__init__()

    self.pool= nn.MaxPool2d(2,2)
    self.dropout= nn.Dropout(p=0.2)

    self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=4)
    self.conv2 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=4)
    self.conv3 = nn.Conv2d(in_channels=12, out_channels=14, kernel_size=4)
    self.conv4 = nn.Conv2d(in_channels=14, out_channels=16, kernel_size=4)
    self.conv5 = nn.Conv2d(in_channels=16, out_channels=20, kernel_size=4)

    self.fc1 = nn.Linear(in_features= 20*4*4, out_features=250)
    self.fc2 = nn.Linear( in_features=   250, out_features=200)
    self.fc3 = nn.Linear( in_features=   200, out_features= 50)
    self.fc4 = nn.Linear( in_features=    50, out_features= 10)
    self.fc5 = nn.Linear( in_features=    10, out_features=  2)

  def forward(self,x):
    x=self.pool(F.relu(self.conv1(x)))
    x=self.pool(F.relu(self.conv2(x)))
    x=self.pool(F.relu(self.conv3(x)))
    x=self.pool(F.relu(self.conv4(x)))
    x=self.pool(F.relu(self.conv5(x)))

    x= x.reshape(-1, 20*4*4)
    x=self.dropout(F.relu(self.fc1(x)))
    x=self.dropout(F.relu(self.fc2(x)))
    x=self.dropout(F.relu(self.fc3(x)))
    x=self.dropout(F.relu(self.fc4(x)))
    x=self.fc5(x)
    return x

device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net= Model().to(device)

print(net)

Model(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (conv1): Conv2d(3, 6, kernel_size=(4, 4), stride=(1, 1))
  (conv2): Conv2d(6, 12, kernel_size=(4, 4), stride=(1, 1))
  (conv3): Conv2d(12, 14, kernel_size=(4, 4), stride=(1, 1))
  (conv4): Conv2d(14, 16, kernel_size=(4, 4), stride=(1, 1))
  (conv5): Conv2d(16, 20, kernel_size=(4, 4), stride=(1, 1))
  (fc1): Linear(in_features=320, out_features=250, bias=True)
  (fc2): Linear(in_features=250, out_features=200, bias=True)
  (fc3): Linear(in_features=200, out_features=50, bias=True)
  (fc4): Linear(in_features=50, out_features=10, bias=True)
  (fc5): Linear(in_features=10, out_features=2, bias=True)
)


In [ ]:
# optimiser and loss function
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
net.train()
for epoch in range(15):
  total_correct= 0.0
  running_loss= 0.0
  for i, (inputs, labels) in enumerate(train_set):
    inputs, labels = inputs.to(device), labels.to(device)
    output= net(inputs)
    output_idx= torch.argmax(output, dim=1)
    total_correct+= (labels== output_idx).sum().item()
    optimizer.zero_grad()
    loss= criterion(output, labels)
    running_loss+= loss.item()*inputs.size(0)
    loss.backward()
    optimizer.step()
  print(f'Epoch:{epoch} Accuracy:{(total_correct/train_len)*100}% loss:{running_loss/train_len}')
print('finished training')

Epoch:0 Accuracy:66.95758258258259% loss:0.6145135154863736
Epoch:1 Accuracy:74.82169669669669% loss:0.5781273439347565
Epoch:2 Accuracy:74.84984984984985% loss:0.5698069869666487
Epoch:3 Accuracy:74.89677177177178% loss:0.5618545686652711
Epoch:4 Accuracy:74.88738738738738% loss:0.5430592953845546
Epoch:5 Accuracy:74.88738738738738% loss:0.5261698836634109
Epoch:6 Accuracy:74.95307807807808% loss:0.5262678896969145
Epoch:7 Accuracy:75.31906906906907% loss:0.5140275155996775
Epoch:8 Accuracy:77.27102102102103% loss:0.500757364002434
Epoch:9 Accuracy:77.32732732732732% loss:0.49646691308991686
Epoch:10 Accuracy:77.45870870870871% loss:0.4847792976790362
Epoch:11 Accuracy:78.32207207207207% loss:0.4775425170858701
Epoch:12 Accuracy:78.68806306306307% loss:0.47149489054808746
Epoch:13 Accuracy:79.76726726726727% loss:0.4581744174610029


In [ ]:
#testing our model
with torch.no_grad():
  net.eval()
  total_loss=0.0
  total_correct=0.0

  for inputs, labels in test_set:
    labels=labels.to(device)
    outputs= net(inputs.to(device))
    loss= criterion(outputs, labels)
    total_loss+= loss.item()*inputs.size(0)
    output_idx= torch.argmax(outputs, dim=1)
    total_correct+= sum(output_idx==labels)

print(f'Accuracy:{(total_correct/test_len)*100}% loss:{total_loss/test_len}')

Accuracy:80.29999542236328% loss:0.4324636409680049


In [ ]:
torch.save(net.state_dict(), 'cat_vs_dog.pt')

In [ ]:
with torch.no_grad():
  model = Model().to(device)
  model.load_state_dict(torch.load('cat_vs_dog.pt'))
  model.eval()

  total_correct = 0.0

  for inputs, labels in test_set:
    labels= labels.to(device)
    outputs= model(inputs.to(device))
    output_idx= torch.argmax(outputs, dim=1)
    total_correct += sum(labels==output_idx)
  print(f'Accuracy: {(total_correct/test_len)*100}%')

Accuracy: 81.21665954589844%


In [ ]:
img= Image.open('./dog.jpg')
img= transform(img).unsqueeze(dim=0).to(device)
prediction= net(img)

print(torch.argmax(prediction))

NameError: name 'Image' is not defined